In [1]:
import os

import numpy as np
import pandas as pd

raw_data_dir = "/home/youngjins/project/belief_trading/data/binance/futures/um/monthly/aggTrades/BTCUSDT/"

# 데이터 파일 목록 가져오기
data_files = [f for f in os.listdir(raw_data_dir) if f.endswith(".csv")]

# 데이터 파일 목록을 날짜 순으로 정렬
data_files.sort()

data_files

['BTCUSDT-aggTrades-2022-04.csv',
 'BTCUSDT-aggTrades-2022-05.csv',
 'BTCUSDT-aggTrades-2022-06.csv',
 'BTCUSDT-aggTrades-2022-07.csv',
 'BTCUSDT-aggTrades-2022-08.csv',
 'BTCUSDT-aggTrades-2022-09.csv',
 'BTCUSDT-aggTrades-2022-10.csv',
 'BTCUSDT-aggTrades-2022-11.csv',
 'BTCUSDT-aggTrades-2022-12.csv',
 'BTCUSDT-aggTrades-2023-01.csv',
 'BTCUSDT-aggTrades-2023-02.csv',
 'BTCUSDT-aggTrades-2023-03.csv',
 'BTCUSDT-aggTrades-2023-04.csv',
 'BTCUSDT-aggTrades-2023-05.csv',
 'BTCUSDT-aggTrades-2023-06.csv',
 'BTCUSDT-aggTrades-2023-07.csv',
 'BTCUSDT-aggTrades-2023-08.csv',
 'BTCUSDT-aggTrades-2023-09.csv',
 'BTCUSDT-aggTrades-2023-10.csv',
 'BTCUSDT-aggTrades-2023-11.csv',
 'BTCUSDT-aggTrades-2023-12.csv',
 'BTCUSDT-aggTrades-2024-01.csv',
 'BTCUSDT-aggTrades-2024-02.csv',
 'BTCUSDT-aggTrades-2024-03.csv',
 'BTCUSDT-aggTrades-2024-04.csv',
 'BTCUSDT-aggTrades-2024-05.csv',
 'BTCUSDT-aggTrades-2024-06.csv',
 'BTCUSDT-aggTrades-2024-07.csv',
 'BTCUSDT-aggTrades-2024-08.csv',
 'BTCUSDT-aggT

In [2]:
# data_files의 데이터 전처리 계획:
# 1. 시간 윈도우: 1시간(1h) 단위로 거래 데이터를 그룹화
# 2. 트레이더 분류: 거래량(quantity) * 가격 (price) 기준으로 상위 20%는 기관 투자자, 나머지 80%는 개인 투자자로 분류
# 3. 거래 방향 정의: 'is_buyer_maker' 필드가 false인 경우 매수(long), true인 경우 매도(short)로 분류
# 4. 집계 방법: 각 시간 윈도우 내에서 트레이더 유형별(개인/기관)로 매수/매도 행동 분포를 계산하여 하나의 데이터 포인트로 생성

# window_size
window_size = 1 * 60 * 60 * 1000  # 1시간(1h)을 밀리초로 변환
type_threshold = 0.2
usecols = [
    "price",
    "quantity",
    "transact_time",
    "is_buyer_maker",
]
for file in data_files:
    df = pd.read_csv(
        os.path.join(raw_data_dir, file),
        index_col=0,
        usecols=usecols,
        nrows=10000000,
    ).reset_index()
    df.loc[:, "transact_time"] = pd.to_datetime(df.loc[:, "transact_time"], unit="ms")

    # 시간 윈도우 설정

    break

In [20]:
# df를 1시간 단위로 그룹화 - 1시간 내에 발생한 것 중에서 거래량 * 가격으로 개인/기관으로 분류 후, long/short 행동 분포를 list로 분류
df.loc[:, "quantity_price"] = df.loc[:, "quantity"] * df.loc[:, "price"]

# 결과 데이터 프레임 생성
# - index: df.index를 1시간 단위로 그룹화한 것
# - columns: retail_long, retail_short, institutional_long, institutional_short
result_start = (df.iloc[0]["transact_time"] + pd.Timedelta(hours=1)).strftime("%Y-%m-%d %H")
result_end = (df.iloc[-1]["transact_time"] + pd.Timedelta(hours=1)).strftime("%Y-%m-%d %H")

result_df = pd.DataFrame(
    index=pd.date_range(
        start=result_start,
        end=result_end,
        freq=f"{window_size}ms",
    ),
    columns=["retail_long", "retail_short", "institutional_long", "institutional_short"],
)

# df의 한시간 단위로 for loop 돌리기
for pivot_time in result_df.index:
    
    df_subset = df.loc[(df.loc[:, "transact_time"] < pivot_time) & (df.loc[:, "transact_time"] >= pivot_time - pd.Timedelta(hours=1))]
    
    if df_subset.empty:
        continue
    
    # 값 정규화를 위한 df_subset의 min, max 값 -> 나중에 mid price로 사용해도 됨
    subset_min = df_subset["quantity_price"].min()
    subset_max = df_subset["quantity_price"].max()
    
    # 개인, 기관 투자자 유형 분류
    # 기관 투자자: 거래량 * 가격 기준 상위 20%
    # 개인 투자자: 거래량 * 가격 기준 하위 80%
    institutional_df = df_subset[df_subset["quantity_price"] > df_subset["quantity_price"].quantile(type_threshold)]
    retail_df = df_subset[df_subset["quantity_price"] <= df_subset["quantity_price"].quantile(type_threshold)]
    
    # long/short 값 분포 계산
    # retail_long: 개인 투자자의 매수(long) 포지션 값 분포
    # retail_short: 개인 투자자의 매도(short) 포지션 값 분포
    retail_long = retail_df[retail_df["is_buyer_maker"] == False]["quantity_price"].tolist()
    retail_short = retail_df[retail_df["is_buyer_maker"] == True]["quantity_price"].tolist()
    
    # 기관 투자자의 매수(long) 포지션 값 분포
    # 기관 투자자의 매도(short) 포지션 값 분포
    institutional_long = institutional_df[institutional_df["is_buyer_maker"] == False]["quantity_price"].tolist()
    institutional_short = institutional_df[institutional_df["is_buyer_maker"] == True]["quantity_price"].tolist()
    
    # 모든 포지션 값들을 subset_min, subset_max 값으로 정규화 (모든 값들을 소수점 4자리에서 반올림)
    # retail_long = [round((x - subset_min) / (subset_max - subset_min), 4) for x in retail_long]
    # retail_short = [round((x - subset_min) / (subset_max - subset_min), 4) for x in retail_short]
    # institutional_long = [round((x - subset_min) / (subset_max - subset_min), 4) for x in institutional_long]
    # institutional_short = [round((x - subset_min) / (subset_max - subset_min), 4) for x in institutional_short]
    
    # # 각각의 리스트를 정렬해서, result_df에 저장
    # retail_long.sort()
    # retail_short.sort()
    # institutional_long.sort()
    # institutional_short.sort()
    
    # result_df.loc[pivot_time, "retail_long"] = retail_long
    # result_df.loc[pivot_time, "retail_short"] = retail_short
    # result_df.loc[pivot_time, "institutional_long"] = institutional_long
    # result_df.loc[pivot_time, "institutional_short"] = institutional_short
    
    break

In [22]:
df_subset["quantity_price"].quantile(type_threshold)

136.74012

In [19]:
retail_df[retail_df["is_buyer_maker"] == True]["quantity_price"].tolist()

[136.5123,
 136.5138,
 45.5048,
 45.5048,
 45.501599999999996,
 45.5013,
 45.4989,
 136.4955,
 45.501599999999996,
 136.4967,
 45.4989,
 45.496900000000004,
 90.99380000000001,
 136.4907,
 45.5041,
 91.0082,
 45.5041,
 91.0082,
 91.0082,
 136.5123,
 91.00880000000001,
 45.5126,
 45.513,
 136.554,
 45.517900000000004,
 91.0428,
 136.5708,
 136.5711,
 45.5315,
 136.5945,
 136.6047,
 45.535199999999996,
 91.07039999999999,
 136.6428,
 45.5546,
 45.5559,
 136.662,
 45.551300000000005,
 45.5508,
 45.5432,
 91.0864,
 45.544900000000005,
 45.544900000000005,
 91.0866,
 91.09060000000001,
 91.0864,
 45.5431,
 45.540800000000004,
 45.5381,
 45.5424,
 45.5421,
 45.5421,
 45.5472,
 136.64159999999998,
 45.5454,
 45.5444,
 136.63320000000002,
 91.0888,
 91.0862,
 45.542500000000004,
 91.0836,
 45.539300000000004,
 45.5372,
 91.08080000000001,
 91.07860000000001,
 91.07480000000001,
 45.5373,
 45.539500000000004,
 45.541000000000004,
 45.5411,
 45.5409,
 45.5342,
 91.0664,
 45.5331,
 91.06,
 45.529